# Control Matrix Asset Allocation Optimisation — Dynamic Wealth Bounds

Optimises a 2D control-matrix policy using `ControlMatrixDynamicBoundsPolicy`:
allocation is a bilinearly interpolated function of time and wealth, where the
wealth grid shifts at each time node.

**Wealth bound construction:**  
For each return sampler, a baseline simulation using 100% equity is run.  
The p90 of the resulting wealth distribution at each timestep is used as the
**upper bound** at that time node.  The **lower bound** is fixed at \$100k.  
Intermediate nodes are arranged in logspace between lower and upper bounds.


In [1]:
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import torch
import sys
import math

np.random.seed(42)
torch.manual_seed(42)

sys.path.append('..')

from utils.experiment import (
    SimulationConfig,
    OptimizationResult,
    save_experiment,
    load_experiments,
)

from utils import (
    CholeskyBootstrapReturns,
    BlockBootstrapReturnsLoader,
    ConstantAllocation,
    ControlMatrixDynamicBoundsPolicy,
    SigmoidWealthPenalty,
    simulate_wealth_trajectory,
    project_onto_simplex,
    project_policy_gradients_tangent_cone,
)
from utils.spending import (
    EXPENDITURE_GUIDELINES,
    DecliningRealSpending,
    DecliningRealFloor,
    NZSuper,
    SpendingPolicy,
)

sns.set_style('whitegrid')
plt.rcParams['figure.figsize'] = (14, 6)

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
if DEVICE.type == 'cuda':
    torch.cuda.empty_cache()
    torch.backends.cudnn.benchmark = True
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"GPU Memory: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")
else:
    print("Running on CPU")


GPU: NVIDIA GeForce RTX 3070
GPU Memory: 8.6 GB


## Configuration Parameters

In [2]:
# ============================================================================
# SIMULATION PARAMETERS
# ============================================================================
N_ASSETS = 3
N_SIMULATIONS = 400_000
SIMULATION_YEARS = 35

# ============================================================================
# CONTROL MATRIX STRUCTURE
# ============================================================================
TIME_NODE_COUNTS = [2, 3, 6, 10]
WEALTH_NODE_COUNTS = [2, 3, 6, 10]

# Fixed lower wealth bound for all time nodes.
LOWER_WEALTH_BOUND = 100_000

# ============================================================================
# RETURN SAMPLER
# ============================================================================
RETURN_SAMPLERS = [
    "cholesky",
    "block_bootstrapped",
    "block_bootstrapped_1950",
]

# ============================================================================
# WEALTH & SPENDING PARAMETERS
# ============================================================================
# Distributed initial wealth buckets [500_000, 550_000, ..., 1_000_000].
INITIAL_WEALTH_MIN = 500_000
INITIAL_WEALTH_MAX = 1_000_000
WEALTH_STEP = 50_000

DESIRED_SPENDING = EXPENDITURE_GUIDELINES['choices_metro_couple']
SPENDING_DECLINE_RATE = 0.02
CONSUMPTION_FLOOR = EXPENDITURE_GUIDELINES['no_frills_metro_couple']
FLOOR_DECLINE_RATE = 0.00
INCOME_TYPE = "couple"  # "couple" | "single" | "single_sharing" | None

# ============================================================================
# OPTIMIZATION PARAMETERS
# ============================================================================
INITIAL_NODE_POLICY = [0.0, 0.0]

OPTIMIZER_TYPE = "ADAM"
INITIAL_LR = 0.0050
MIN_LR = 1e-4
MAX_ITERATIONS = 20_000
MOMENTUM = 0.9

BATCH_SIZE_SCHEDULE = [
    # size, learning_rate, lr_patience, min_iterations
    (50_000,  0.050,  150, 150),
    (50_000,  0.005,  150, 150),
    (100_000, 0.0015, 150, 150),
    (100_000, 0.001,  100, 100),
    (400_000, 0.001,   40, None),
]

LR_DECAY_FACTOR = 0.25
LR_PATIENCE = 40
LR_THRESHOLD = 1e-4

STOPPING_PATIENCE = 60

PRINT_EVERY = 5
HISTORY_SAVE_FREQUENCY = 20

# ============================================================================
# OBJECTIVE FUNCTION PARAMETERS
# ============================================================================
WEALTH_PENALTY_STEEPNESS = 1e-3

# ============================================================================
# RESULTS DIRECTORY
# ============================================================================
RESULTS_DIR = "Results_Dynamic"


## Load Data

In [3]:
tax_rates = np.loadtxt("../Data/IID Data/Final/tax_rates.csv", delimiter=",", skiprows=1)

return_sampler_loaders = {}
for sampler_name in RETURN_SAMPLERS:
    if sampler_name == "cholesky":
        exp_returns = np.loadtxt("../Data/IID Data/Final/expected_returns.csv", delimiter=",", skiprows=1)
        cov_matrix = np.loadtxt(
            "../Data/IID Data/Final/covariance.csv", delimiter=",", skiprows=1, usecols=range(1, N_ASSETS + 2)
        )
        return_sampler_loaders[sampler_name] = CholeskyBootstrapReturns(exp_returns, cov_matrix)
    elif sampler_name in {"block_bootstrapped", "block_bootstrapped_1950"}:
        return_sampler_loaders[sampler_name] = BlockBootstrapReturnsLoader(
            f"../Data/Returns/Final/{sampler_name}.npy"
        )
    else:
        raise ValueError(f"Unknown RETURN_SAMPLER: {sampler_name}")


## Setup

In [4]:
completed_runs = set()
try:
    existing_experiments = load_experiments(results_dir=RESULTS_DIR)
    for payload in existing_experiments:
        cfg = payload["config"]
        completed_runs.add((cfg.RETURN_SAMPLER, int(cfg.TIME_NODE_COUNT), int(cfg.WEALTH_NODE_COUNT)))
    print(f"Loaded {len(completed_runs)} completed runs from {RESULTS_DIR}/.")
except FileNotFoundError:
    print(f"{RESULTS_DIR}/ not found yet - all runs will be executed.")
except Exception as exc:
    print(f"Could not read existing {RESULTS_DIR} ({exc}); proceeding without skip cache.")


def _build_dynamic_wealth_nodes(
    p90_over_sim_time: np.ndarray,
    time_nodes: torch.Tensor,
    wealth_node_count: int,
    lower_bound: float,
    simulation_years: int,
) -> torch.Tensor:
    """Build a (TIME_NODE_COUNT x WEALTH_NODE_COUNT) wealth-node matrix.

    At each time node, interpolates the p90 wealth from the 100%-equity
    baseline run to get the upper bound, then arranges ``wealth_node_count``
    nodes in logspace between ``lower_bound`` and that upper bound.

    Parameters
    ----------
    p90_over_sim_time : np.ndarray, shape (simulation_years + 1,)
        P90 wealth at each integer year [0, 1, ..., simulation_years].
    time_nodes : torch.Tensor, shape (TIME_NODE_COUNT,)
        Time points at which nodes are required.
    wealth_node_count : int
    lower_bound : float
    simulation_years : int

    Returns
    -------
    torch.Tensor, shape (TIME_NODE_COUNT, wealth_node_count)
    """
    sim_time_axis = np.arange(0, simulation_years + 1, dtype=np.float64)
    upper_bounds = np.interp(time_nodes.numpy().astype(np.float64), sim_time_axis, p90_over_sim_time)

    rows = []
    for ub in upper_bounds:
        # Ensure upper is meaningfully above lower to keep logspace well-defined.
        upper = max(float(ub), lower_bound * 1.5)
        nodes = torch.logspace(
            math.log10(lower_bound),
            math.log10(upper),
            wealth_node_count,
        )
        rows.append(nodes)

    return torch.stack(rows)  # (TIME_NODE_COUNT, wealth_node_count)


def _build_fixed_batch_indices(total_size: int, batch_size: int, device: torch.device):
    """Create deterministic contiguous batch index chunks for one stage."""
    if batch_size >= total_size:
        return None
    all_indices = torch.arange(total_size, device=device)
    return [
        all_indices[start:start + batch_size]
        for start in range(0, total_size, batch_size)
    ]


wealth_penalty = SigmoidWealthPenalty(steepness=WEALTH_PENALTY_STEEPNESS)

# Distributed initial wealth tensor.
wealth_levels = torch.arange(
    INITIAL_WEALTH_MIN, INITIAL_WEALTH_MAX + WEALTH_STEP, WEALTH_STEP, dtype=torch.float32
)
n_buckets = len(wealth_levels)
repeats = N_SIMULATIONS // n_buckets
remainder = N_SIMULATIONS % n_buckets
initial_wealth_tensor = torch.cat([
    wealth_levels.repeat(repeats),
    wealth_levels[:remainder],
]).to(DEVICE)

print(f"Initial wealth buckets: {wealth_levels.numpy()} ({n_buckets} levels, ~{repeats:,} sims each)")
print(f"Batch size schedule: {BATCH_SIZE_SCHEDULE}")

# Validate batch schedule
for stage_idx, (stage_batch_size, stage_lr, stage_lr_patience, stage_min_iterations) in enumerate(BATCH_SIZE_SCHEDULE):
    assert stage_batch_size > 0, f"Stage {stage_idx}: non-positive batch size"
    assert stage_batch_size <= N_SIMULATIONS, f"Stage {stage_idx}: batch size exceeds N_SIMULATIONS"
    assert stage_lr_patience > 0, f"Stage {stage_idx}: non-positive lr_patience"
    if stage_min_iterations is not None:
        assert stage_min_iterations > 0, f"Stage {stage_idx}: non-positive min_iterations"


Results_Dynamic/ not found yet - all runs will be executed.
Initial wealth buckets: [5.0e+05 5.5e+05 6.0e+05 6.5e+05 7.0e+05 7.5e+05 8.0e+05 8.5e+05 9.0e+05
 9.5e+05 1.0e+06] (11 levels, ~36,363 sims each)
Batch size schedule: [(50000, 0.05, 150, 150), (50000, 0.005, 150, 150), (100000, 0.0015, 150, 150), (100000, 0.001, 100, 100), (400000, 0.001, 40, None)]


## Optimisation

In [5]:
from IPython.display import clear_output
from wakepy import keep

with keep.running():
    for RETURN_SAMPLER in RETURN_SAMPLERS:
        sampler = return_sampler_loaders[RETURN_SAMPLER]
        returns_sample, cumulative_inflation_sample = sampler.generate(N_SIMULATIONS, SIMULATION_YEARS)

        returns = torch.tensor(returns_sample - tax_rates, device=DEVICE)
        cumulative_inflation = torch.tensor(cumulative_inflation_sample, device=DEVICE)

        # --------------------------------------------------------------------
        # Compute p90 wealth trajectory under 100% equity allocation.
        # This is used to set the upper wealth bound at each time node.
        # --------------------------------------------------------------------
        print(f"\nComputing 100%-equity baseline for sampler={RETURN_SAMPLER}...")

        desired_spending_pol_baseline = DecliningRealSpending(DESIRED_SPENDING, decline_rate=SPENDING_DECLINE_RATE)
        consumption_floor_pol_baseline = DecliningRealFloor(init_floor=CONSUMPTION_FLOOR, decline_rate=FLOOR_DECLINE_RATE)
        income_baseline = NZSuper(INCOME_TYPE)
        spending_policy_baseline = SpendingPolicy(
            spending=desired_spending_pol_baseline,
            floor=consumption_floor_pol_baseline,
            income=income_baseline,
        )

        equity_policy = ConstantAllocation(N_ASSETS, N_SIMULATIONS, DEVICE)
        # 100% equities: bonds=0, stocks=1 (safe asset receives residual = 0)
        equity_policy_settings = torch.tensor([0.0, 1.0], device=DEVICE)

        with torch.no_grad():
            wealth_equity, _ = simulate_wealth_trajectory(
                returns=returns,
                cumulative_inflation=cumulative_inflation,
                allocation_policy=equity_policy,
                spending_policy=spending_policy_baseline,
                initial_wealth=initial_wealth_tensor,
                policy_settings=equity_policy_settings,
            )

        # wealth_equity: (N_SIMULATIONS, SIMULATION_YEARS + 1), includes t=0
        p90_over_sim_time = torch.quantile(
            wealth_equity.cpu().float(), 0.9, dim=0
        ).numpy()  # shape: (SIMULATION_YEARS + 1,)

        print(f"  p90 wealth at t=0:  ${p90_over_sim_time[0]:,.0f}")
        print(f"  p90 wealth at t={SIMULATION_YEARS}: ${p90_over_sim_time[-1]:,.0f}")

        del wealth_equity  # free GPU memory

        # --------------------------------------------------------------------
        # Sweep over node counts
        # --------------------------------------------------------------------
        for WEALTH_NODE_COUNT in WEALTH_NODE_COUNTS:
            for TIME_NODE_COUNT in TIME_NODE_COUNTS:
                run_key = (RETURN_SAMPLER, int(TIME_NODE_COUNT), int(WEALTH_NODE_COUNT))
                if run_key in completed_runs:
                    print(
                        f"Skipping completed run: sampler={RETURN_SAMPLER}, "
                        f"TIME_NODE_COUNT={TIME_NODE_COUNT}, WEALTH_NODE_COUNT={WEALTH_NODE_COUNT}"
                    )
                    continue

                time_nodes = torch.linspace(0, SIMULATION_YEARS, steps=TIME_NODE_COUNT)

                wealth_nodes = _build_dynamic_wealth_nodes(
                    p90_over_sim_time=p90_over_sim_time,
                    time_nodes=time_nodes,
                    wealth_node_count=WEALTH_NODE_COUNT,
                    lower_bound=LOWER_WEALTH_BOUND,
                    simulation_years=SIMULATION_YEARS,
                )
                
                print(f"\n{'='*70}")
                print(
                    f"STARTING: sampler={RETURN_SAMPLER}, "
                    f"TIME_NODE_COUNT={TIME_NODE_COUNT}, WEALTH_NODE_COUNT={WEALTH_NODE_COUNT}"
                )
                print(f"Time nodes: {time_nodes.numpy()}")
                print("Wealth nodes (lower / upper per time node):")
                for ti in range(TIME_NODE_COUNT):
                    print(f"  t={time_nodes[ti]:.1f}: ${wealth_nodes[ti, 0]:,.0f} .. ${wealth_nodes[ti, -1]:,.0f}")
                print(f"{'='*70}")

                sim_config = SimulationConfig(
                    N_ASSETS=N_ASSETS,
                    N_SIMULATIONS=N_SIMULATIONS,
                    SIMULATION_YEARS=SIMULATION_YEARS,
                    RETURN_SAMPLER=RETURN_SAMPLER,
                    INITIAL_WEALTH=None,
                    DESIRED_SPENDING=DESIRED_SPENDING,
                    SPENDING_DECLINE_RATE=SPENDING_DECLINE_RATE,
                    CONSUMPTION_FLOOR=CONSUMPTION_FLOOR,
                    FLOOR_DECLINE_RATE=FLOOR_DECLINE_RATE,
                    INCOME_TYPE=INCOME_TYPE,
                    INITIAL_POLICY=INITIAL_NODE_POLICY,
                    OPTIMIZER_TYPE=OPTIMIZER_TYPE,
                    TIME_NODE_COUNT=TIME_NODE_COUNT,
                    WEALTH_NODE_COUNT=WEALTH_NODE_COUNT,
                    USES_DYNAMIC_WEALTH_BOUNDS=True,
                )

                desired_spending_pol = DecliningRealSpending(DESIRED_SPENDING, decline_rate=SPENDING_DECLINE_RATE)
                consumption_floor_pol = DecliningRealFloor(init_floor=CONSUMPTION_FLOOR, decline_rate=FLOOR_DECLINE_RATE)
                income = NZSuper(INCOME_TYPE)
                spending_policy = SpendingPolicy(
                    spending=desired_spending_pol,
                    floor=consumption_floor_pol,
                    income=income,
                )

                allocation_policy = ControlMatrixDynamicBoundsPolicy(
                    N_ASSETS,
                    N_SIMULATIONS,
                    time_nodes.clone(),
                    wealth_nodes.clone(),
                    DEVICE,
                )

                # Initialise policy at zeros.
                base_node = torch.tensor(INITIAL_NODE_POLICY, device=DEVICE, dtype=torch.float32)
                policy = (
                    base_node
                    .unsqueeze(0)
                    .unsqueeze(0)
                    .repeat(TIME_NODE_COUNT, WEALTH_NODE_COUNT, 1)
                    .requires_grad_(True)
                )

                # Set up optimizer and batch schedule.
                batch_stage_idx = 0
                current_batch_size, scheduled_lr, current_lr_patience, current_min_iterations = BATCH_SIZE_SCHEDULE[batch_stage_idx]
                stage_start_iteration = 0
                stage_batch_chunks = _build_fixed_batch_indices(N_SIMULATIONS, current_batch_size, DEVICE)
                stage_batch_ptr = 0

                if OPTIMIZER_TYPE.upper() == "ADAM":
                    optimizer = torch.optim.Adam([policy], lr=scheduled_lr)
                else:
                    optimizer = torch.optim.SGD([policy], lr=scheduled_lr, momentum=MOMENTUM)

                cost_history = []
                policy_history = []
                best_cost = float('inf')
                best_policy = policy.data.clone()
                iterations_without_improvement = 0
                iterations_without_lr_improvement = 0

                for i in range(MAX_ITERATIONS):
                    optimizer.zero_grad()

                    if current_batch_size < N_SIMULATIONS:
                        batch_idx = stage_batch_chunks[stage_batch_ptr]
                        stage_batch_ptr = (stage_batch_ptr + 1) % len(stage_batch_chunks)
                        batch_returns = returns[batch_idx]
                        batch_cumulative_inflation = cumulative_inflation[batch_idx]
                        batch_initial_wealth = initial_wealth_tensor[batch_idx]
                    else:
                        batch_returns = returns
                        batch_cumulative_inflation = cumulative_inflation
                        batch_initial_wealth = initial_wealth_tensor

                    wealth, consumption = simulate_wealth_trajectory(
                        returns=batch_returns,
                        cumulative_inflation=batch_cumulative_inflation,
                        allocation_policy=allocation_policy,
                        spending_policy=spending_policy,
                        initial_wealth=batch_initial_wealth,
                        policy_settings=policy,
                    )

                    cost = wealth_penalty.evaluate(wealth, consumption)
                    cost.backward()
                    project_policy_gradients_tangent_cone(policy)
                    optimizer.step()

                    with torch.no_grad():
                        policy.data = project_onto_simplex(policy.data).clamp(0, 1)

                    cost_item = cost.item()
                    cost_history.append(cost_item)

                    if (i + 1) % HISTORY_SAVE_FREQUENCY == 0:
                        policy_history.append(policy.data.cpu().numpy().copy())

                    if cost_item < best_cost:
                        best_cost = cost_item
                        best_policy = policy.detach().clone().cpu()
                        iterations_without_improvement = 0
                        iterations_without_lr_improvement = 0
                    else:
                        iterations_without_improvement += 1
                        iterations_without_lr_improvement += 1

                    if iterations_without_improvement >= STOPPING_PATIENCE and batch_stage_idx >= len(BATCH_SIZE_SCHEDULE) - 1:
                        print(f"\nEARLY STOPPING at iteration {i + 1} (no improvement for {STOPPING_PATIENCE} iterations)")
                        break

                    stage_iterations = (i + 1) - stage_start_iteration
                    min_iterations_reached = (
                        current_min_iterations is None or stage_iterations >= current_min_iterations
                    )
                    if iterations_without_lr_improvement >= current_lr_patience and min_iterations_reached:
                        used_batch_schedule_step = False

                        if batch_stage_idx < len(BATCH_SIZE_SCHEDULE) - 1:
                            batch_stage_idx += 1
                            current_batch_size, scheduled_lr, current_lr_patience, current_min_iterations = BATCH_SIZE_SCHEDULE[batch_stage_idx]
                            for param_group in optimizer.param_groups:
                                old_lr = param_group['lr']
                                param_group['lr'] = scheduled_lr
                            print(
                                f"\nBatch schedule advanced at iteration {i + 1}: "
                                f"stage {batch_stage_idx + 1}/{len(BATCH_SIZE_SCHEDULE)}, "
                                f"batch={current_batch_size:,}, lr={old_lr:.6f}->{scheduled_lr:.6f}"
                            )
                            used_batch_schedule_step = True
                            iterations_without_lr_improvement = 0
                            iterations_without_improvement = 0
                            best_cost = float('inf')
                            stage_start_iteration = i + 1
                            stage_batch_chunks = _build_fixed_batch_indices(N_SIMULATIONS, current_batch_size, DEVICE)
                            stage_batch_ptr = 0
                        else:
                            stop = False
                            for param_group in optimizer.param_groups:
                                old_lr = param_group['lr']
                                new_lr = old_lr * LR_DECAY_FACTOR
                                param_group['lr'] = new_lr
                                if new_lr < MIN_LR:
                                    print(f"\nLR below minimum at iteration {i + 1}: {new_lr:.6f} < {MIN_LR:.6f}. Stopping.")
                                    stop = True
                                    break
                            if stop:
                                break
                            print(f"LR reduced at iteration {i + 1}: {old_lr:.6f} -> {new_lr:.6f}")
                            iterations_without_lr_improvement = 0

                        if used_batch_schedule_step:
                            continue

                    if (i + 1) % PRINT_EVERY == 0 or i == 0:
                        clear_output(wait=True)
                        current_lr = optimizer.param_groups[0]['lr']
                        cost_change = cost_history[-1] - cost_history[-2] if len(cost_history) > 1 else 0
                        stage_label = f"{batch_stage_idx + 1}/{len(BATCH_SIZE_SCHEDULE)}"
                        batch_size_for_metrics = wealth.shape[0]

                        print(f"{'='*70}")
                        print(
                            f"OPTIMISING: sampler={RETURN_SAMPLER} | "
                            f"TIME_NODES={TIME_NODE_COUNT} | WEALTH_NODES={WEALTH_NODE_COUNT}"
                        )
                        print(
                            f"Iter {i+1:,}/{MAX_ITERATIONS:,}  |  "
                            f"LR: {current_lr:.6f}  |  Batch: {current_batch_size:,} (stage {stage_label})"
                        )
                        print(f"{'='*70}")
                        print(f"Cost:                   {cost_item:.6f}")
                        print(f"Cost change:            {cost_change:.8f}")
                        print(f"Best cost:              {best_cost:.6f}")
                        print(f"Iters w/o improve:      {iterations_without_improvement}/{STOPPING_PATIENCE}")
                        print(f"{'-'*70}")
                        print(f"Mean consumption:       ${consumption.mean().item():,.0f}")
                        print(f"Mean terminal wealth:   ${wealth[:, -1].mean().item():,.0f}")
                        print(f"Bankruptcy rate:        {(wealth[:, -1] == 0).sum().item() / batch_size_for_metrics:.2%}")
                        bankruptcy_density = (wealth == 0).sum().item() / (batch_size_for_metrics * SIMULATION_YEARS)
                        floor_t = consumption_floor_pol.calculate_tensor(
                            wealth=wealth, cumulative_inflation=batch_cumulative_inflation
                        )
                        impoverishment_density = (consumption < floor_t).sum().item() / (batch_size_for_metrics * SIMULATION_YEARS)
                        print(f"Impoverishment density: {impoverishment_density:.4%}")
                        print(f"Bankruptcy density:     {bankruptcy_density:.4%}")
                        print(f"{'-'*70}")
                        print("Current policy (% bonds / % stocks):")
                        mid_w_idx = WEALTH_NODE_COUNT // 2
                        for t_idx in range(TIME_NODE_COUNT):
                            age = int(time_nodes[t_idx].item()) + 65
                            lo_b  = policy[t_idx,  0,       0].item()
                            lo_s  = policy[t_idx,  0,       1].item()
                            mid_b = policy[t_idx,  mid_w_idx, 0].item()
                            mid_s = policy[t_idx,  mid_w_idx, 1].item()
                            hi_b  = policy[t_idx, -1,       0].item()
                            hi_s  = policy[t_idx, -1,       1].item()
                            print(
                                f"  Age {age:3d}: "
                                f"lowW {lo_b:.2%}/{lo_s:.2%}  "
                                f"midW {mid_b:.2%}/{mid_s:.2%}  "
                                f"highW {hi_b:.2%}/{hi_s:.2%}"
                            )
                        print(f"{'='*70}")

                # Restore best policy and run final full simulation.
                with torch.no_grad():
                    policy.copy_(best_policy.to(policy.device))

                policy_history = np.array(policy_history)
                cost_history = np.array(cost_history)

                print(f"\n{'='*70}")
                print(
                    f"COMPLETE: sampler={RETURN_SAMPLER} "
                    f"| TIME_NODES={TIME_NODE_COUNT} | WEALTH_NODES={WEALTH_NODE_COUNT}"
                )
                print(f"Best cost: {best_cost:.6f}  |  Iterations: {len(cost_history):,}")
                print(f"{'='*70}")

                final_wealth, final_consumption = simulate_wealth_trajectory(
                    returns=returns,
                    cumulative_inflation=cumulative_inflation,
                    allocation_policy=allocation_policy,
                    spending_policy=spending_policy,
                    initial_wealth=initial_wealth_tensor,
                    policy_settings=policy,
                )

                result = OptimizationResult(
                    best_policy=best_policy.detach().cpu().numpy(),
                    best_utility=-best_cost,
                    policy_history=policy_history,
                    cost_history=cost_history,
                    wealth_simulated=final_wealth.detach().cpu().numpy(),
                    consumption_simulated=final_consumption.detach().cpu().numpy(),
                    cumulative_inflation=cumulative_inflation.detach().cpu().numpy(),
                    time_nodes=time_nodes.detach().cpu().numpy(),
                    # 2D array: (TIME_NODE_COUNT, WEALTH_NODE_COUNT)
                    wealth_nodes=wealth_nodes.detach().cpu().numpy(),
                )
                save_experiment(sim_config, result, results_dir=RESULTS_DIR)
                completed_runs.add(run_key)


OPTIMISING: sampler=block_bootstrapped_1950 | TIME_NODES=10 | WEALTH_NODES=10
Iter 1,830/20,000  |  LR: 0.000250  |  Batch: 400,000 (stage 5/5)
Cost:                   -0.963256
Cost change:            0.00000000
Best cost:              -0.963256
Iters w/o improve:      59/60
----------------------------------------------------------------------
Mean consumption:       $159,707
Mean terminal wealth:   $2,239,859
Bankruptcy rate:        14.53%
Impoverishment density: 7.1180%
Bankruptcy density:     6.1795%
----------------------------------------------------------------------
Current policy (% bonds / % stocks):
  Age  65: lowW 0.00%/100.00%  midW 0.00%/100.00%  highW 0.00%/24.90%
  Age  68: lowW 0.00%/100.00%  midW 0.00%/100.00%  highW 0.00%/89.15%
  Age  72: lowW 0.00%/100.00%  midW 0.00%/62.01%  highW 0.00%/40.27%
  Age  76: lowW 0.00%/100.00%  midW 0.00%/52.34%  highW 0.00%/26.38%
  Age  80: lowW 0.00%/100.00%  midW 0.00%/48.08%  highW 0.05%/86.92%
  Age  84: lowW 0.00%/100.00%  mid

In [6]:
p90_over_sim_time

array([9.5000000e+05, 1.0624054e+06, 1.1683535e+06, 1.2737568e+06,
       1.3900725e+06, 1.5183530e+06, 1.6656104e+06, 1.8163861e+06,
       1.9787258e+06, 2.1570065e+06, 2.3455378e+06, 2.5557338e+06,
       2.7921060e+06, 3.0594930e+06, 3.3487005e+06, 3.6759822e+06,
       4.0289995e+06, 4.4293850e+06, 4.8660010e+06, 5.3378720e+06,
       5.8880645e+06, 6.4725240e+06, 7.1309835e+06, 7.8641945e+06,
       8.6686350e+06, 9.5813820e+06, 1.0579362e+07, 1.1693975e+07,
       1.2950252e+07, 1.4337039e+07, 1.5845613e+07, 1.7476214e+07,
       1.9320346e+07, 2.1416960e+07, 2.3717970e+07, 2.6179774e+07],
      dtype=float32)